In [1]:
from pathlib import Path
from time import perf_counter
import cProfile
import os
import pstats
import numpy as np
import sys

ROOT_DIR = Path.cwd()
if not (ROOT_DIR / "src").exists():
    ROOT_DIR = ROOT_DIR.parent
SRC_ROOT = ROOT_DIR / "src"

sys.path.insert(0, str(SRC_ROOT))
os.environ.setdefault("GRB_LICENSE_FILE", str(ROOT_DIR / "gurobi.lic"))

from PolyRound import main as pr_old   # Packaged
import hopsy._polyround as pr_new      # Vendored


In [2]:
from pathlib import Path
import json
import urllib.request

out_dir = ROOT_DIR / "examples" / "test_data"
out_dir.mkdir(parents=True, exist_ok=True)

# Get all BiGG model IDs
with urllib.request.urlopen("http://bigg.ucsd.edu/api/v2/models") as response: data = json.load(response)

model_ids = [m["bigg_id"] for m in data["results"]]

# Download each model as .xml.gz
for model_id in model_ids:
  url = f"http://bigg.ucsd.edu/static/models/{model_id}.xml.gz"
  target = out_dir / f"{model_id}.xml.gz"

  if target.exists():
    print(f"Skipping {model_id}; already exists at {target}")
    continue

  tmp_target = target.with_name(target.name + ".part")
  tmp_target.unlink(missing_ok=True)

  print(f"Downloading {model_id} > {target}")
  urllib.request.urlretrieve(url, tmp_target)

  tmp_target.replace(target)

Skipping e_coli_core; already exists at /home/edoar/Projects/fzj/hopsy-196/examples/test_data/e_coli_core.xml.gz
Skipping iAB_RBC_283; already exists at /home/edoar/Projects/fzj/hopsy-196/examples/test_data/iAB_RBC_283.xml.gz
Skipping iAF1260; already exists at /home/edoar/Projects/fzj/hopsy-196/examples/test_data/iAF1260.xml.gz
Skipping iAF1260b; already exists at /home/edoar/Projects/fzj/hopsy-196/examples/test_data/iAF1260b.xml.gz
Skipping iAF692; already exists at /home/edoar/Projects/fzj/hopsy-196/examples/test_data/iAF692.xml.gz
Skipping iAF987; already exists at /home/edoar/Projects/fzj/hopsy-196/examples/test_data/iAF987.xml.gz
Skipping iAM_Pb448; already exists at /home/edoar/Projects/fzj/hopsy-196/examples/test_data/iAM_Pb448.xml.gz
Skipping iAM_Pc455; already exists at /home/edoar/Projects/fzj/hopsy-196/examples/test_data/iAM_Pc455.xml.gz
Skipping iAM_Pf480; already exists at /home/edoar/Projects/fzj/hopsy-196/examples/test_data/iAM_Pf480.xml.gz
Skipping iAM_Pk459; already e

In [3]:
import gzip
import shutil
import cobra

MODELS_DIR = ROOT_DIR / "examples" / "test_data"
N_MODELS = 10

MODELS = sorted(
    MODELS_DIR.glob("*.xml.gz"),
    key=lambda p: p.stat().st_size,
)[:N_MODELS]
if not MODELS:
    raise FileNotFoundError(f"No .xml.gz models found in {MODELS_DIR}. Run the download cell first.")

print(f"Selected {len(MODELS)} models sorted by .xml.gz size:")
for path in MODELS:
    print(f"  {path.name:<35} {path.stat().st_size / 1024:>10.1f} KiB")

SETTINGS = {
    "thresh": 1e-7,
    "verbose": False,
    "sgp": False,
    "reduce": True,
    "presolve": True,
    "regularize": False,
    "check_lps": False,
    "hp_flags": {
        "FeasibilityTol": 1e-9,
        "OptimalityTol": 1e-8,
        "Threads": 1,
        # "Method": -1,
    },
}

RUNS = [
    ("pr_old", 
     pr_old, 
     [
         "gurobi",
         # "glpk"
     ]),
    ("pr_new", 
     pr_new, 
     [
         # "gurobi", 
         # "glpk", 
         # "highs", 
         "exp1"
     ])
]


Selected 10 models sorted by .xml.gz size:
  e_coli_core.xml.gz                        34.7 KiB
  iAB_RBC_283.xml.gz                       120.3 KiB
  iIS312.xml.gz                            140.6 KiB
  iIS312_Epimastigote.xml.gz               141.6 KiB
  iIS312_Amastigote.xml.gz                 142.0 KiB
  iIS312_Trypomastigote.xml.gz             143.2 KiB
  iIT341.xml.gz                            144.5 KiB
  iLJ478.xml.gz                            164.5 KiB
  iNF517.xml.gz                            167.5 KiB
  iAF692.xml.gz                            181.8 KiB


In [4]:
def make_settings(pr_module, backend):
    return pr_module.PolyRoundSettings(
        backend=backend,
        hp_flags=dict(SETTINGS["hp_flags"]),
        thresh=SETTINGS["thresh"],
        verbose=SETTINGS["verbose"],
        sgp=SETTINGS["sgp"],
        reduce=SETTINGS["reduce"],
        regularize=SETTINGS["regularize"],
        check_lps=SETTINGS["check_lps"],
        presolve=SETTINGS["presolve"],
    )


def sbml_to_polytope(pr_module, model_path, settings):
    if hasattr(pr_module, "StoichiometryParser"):
        return pr_module.StoichiometryParser.parse_sbml_cobrapy(
            str(model_path),
        )

    return pr_module.PolyRoundApi.sbml_to_polytope(
        str(model_path),
        settings=settings,
    )

def polytope_snapshot(polytope):
    S = getattr(polytope, "S", None)
    h = getattr(polytope, "h", None)
    return {
        "A": np.asarray(polytope.A.values).copy(),
        "b": np.asarray(polytope.b.values).copy(),
        "S": np.asarray([] if S is None else S.values).copy(),
        "h": np.asarray([] if h is None else h.values).copy(),
    }


def append_timing(model_name, pr_name, backend, stage, seconds):
    timings.append(
        {
            "model": model_name,
            "pr": pr_name,
            "backend": backend,
            "stage": stage,
            "seconds": seconds,
        }
    )


RESULTS = {}
timings = []

print()
print(" --- Settings --- ")
print()
for key, value in SETTINGS.items():
    print(f"{key:<16}{value}")
print()
print(f"Selected {len(MODELS)} models from {MODELS_DIR}")
print()

print()
print(" --- Run --- ")
print()

for gz_path in MODELS:
    model_path = gz_path.with_suffix("")  # .xml.gz -> .xml
    model_name = model_path.name

    try:
        print()
        print(f"Extracting {gz_path.name} > {model_name}", flush=True)
        with gzip.open(gz_path, "rb") as src:
            with open(model_path, "wb") as dst:
                shutil.copyfileobj(src, dst)

        for pr_name, pr, backends in RUNS:
            for backend in backends:
                print(f"    Running {model_name} --- {pr_name} - {backend}", flush=True)

                settings = make_settings(pr, backend)
                profiler = cProfile.Profile()
                backend_total = 0.0

                # Parse
                start = perf_counter()
                parsed = sbml_to_polytope(
                    pr,
                    model_path,
                    settings=settings,
                )
                stop = perf_counter()
                parse_elapsed = stop - start
                backend_total += parse_elapsed
                print(f"{'':<8}", f"{'parse':<30}{parse_elapsed:>10.3f}s", flush=True)
                append_timing(model_name, pr_name, backend, "parse", parse_elapsed)
                RESULTS[(model_name, pr_name, backend, "parse")] = polytope_snapshot(parsed)

                # Simplify
                start = perf_counter()
                profiler.enable()
                simplified = pr.PolyRoundApi.simplify_polytope(parsed, settings=settings)
                profiler.disable()
                stop = perf_counter()
                simplify_elapsed = stop - start
                backend_total += simplify_elapsed
                print(f"{'':<8}", f"{'simplify':<30}{simplify_elapsed:>10.3f}s", flush=True)
                append_timing(model_name, pr_name, backend, "simplify", simplify_elapsed)
                RESULTS[(model_name, pr_name, backend, "simplify")] = polytope_snapshot(simplified)

                # Transform
                start = perf_counter()
                profiler.enable()
                round_input = simplified
                if not simplified.inequality_only:
                    round_input = pr.PolyRoundApi.transform_polytope(simplified, settings=settings)
                profiler.disable()
                stop = perf_counter()
                transform_elapsed = stop - start
                backend_total += transform_elapsed
                print(f"{'':<8}", f"{'transform':<30}{transform_elapsed:>10.3f}s", flush=True)
                append_timing(model_name, pr_name, backend, "transform", transform_elapsed)
                RESULTS[(model_name, pr_name, backend, "transform")] = polytope_snapshot(round_input)

                # Round
                start = perf_counter()
                profiler.enable()
                rounded = pr.PolyRoundApi.round_polytope(round_input, settings=settings)
                profiler.disable()
                stop = perf_counter()
                round_elapsed = stop - start
                backend_total += round_elapsed
                print(f"{'':<8}", f"{'round':<30}{round_elapsed:>10.3f}s", flush=True)
                append_timing(model_name, pr_name, backend, "round", round_elapsed)
                print(f"{'':<8}", f"{'total':<30}{backend_total:>10.3f}s", flush=True)
                append_timing(model_name, pr_name, backend, "total", backend_total)
                RESULTS[(model_name, pr_name, backend, "round")] = polytope_snapshot(rounded)

                # print()
                # print(" --- Profiling --- ")
                # print()
                # stats = pstats.Stats(profiler).strip_dirs()
                # stats.sort_stats("tottime").print_stats(5)
                # stats.sort_stats("cumtime").print_stats(5)
                # stats.print_callers(5)
                # stats.print_callees(5)
    
    except Exception as exc:
        print(
            f"FAILED: {model_name} - {pr_name} - {backend}: {type(exc).__name__}: {exc}",
            flush=True,
        )
        
    finally:
        model_path.unlink(missing_ok=True)

print()
print(" --- Timings --- ", flush=True)
print()
print(f"{'model':<30}{'pr':<10}{'backend':<12}{'stage':<12}{'time':>10}")
print()

previous = None
for timing in timings:
    current = (timing["model"], timing["pr"], timing["backend"])
    if previous is not None and current != previous:
        print()
    previous = current
    seconds = f"{timing['seconds']:.3f}s"
    print(
        f"{timing['model']:<30}"
        f"{timing['pr']:<10}"
        f"{timing['backend']:<12}"
        f"{timing['stage']:<12}"
        f"{seconds:>10}"
    )



 --- Settings --- 

thresh          1e-07
verbose         False
sgp             False
reduce          True
presolve        True
regularize      False
check_lps       False
hp_flags        {'FeasibilityTol': 1e-09, 'OptimalityTol': 1e-08, 'Threads': 1}

Selected 10 models from /home/edoar/Projects/fzj/hopsy-196/examples/test_data


 --- Run --- 


Extracting e_coli_core.xml.gz > e_coli_core.xml
    Running e_coli_core.xml --- pr_old - gurobi
Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2783841
Academic license 2783841 - for non-commercial use only - registered to ed___@chalmers.se
         parse                              0.168s
         simplify                           0.769s
         transform                          0.073s
         round                              0.407s
         total                              1.416s
    Running e_coli_core.xml --- pr_new - exp1
         parse                              0.391s
         simplify     